# 階層型GP: 条件によって有効になる変数が変わる場合

このNotebookでは `HierarchicalConditionalKernelGP` と `HierarchicalConditionalKernelMultiTaskGP` を使います。

例として、共通条件 `x` に加えて、プロセス方式によって有効な条件が変わる探索空間を考えます。

- 方式0: 温度が有効、圧力は無効
- 方式1: 圧力が有効、温度は無効

通常のGPでは無効な変数まで距離計算に入ってしまいますが、階層型カーネルでは親変数の値に応じて有効な子変数だけを使って共分散を計算できます。

## このNotebookで学ぶこと

- 通常のMixed GPと**階層型（条件付き）GP**の違い
- 親変数・子変数の有効／無効を `hierarchical_dependencies` で表現する方法
- 枝ごとの事後平均と不確実性の読み方
- Multi-taskのtask IDと、方式を選択する親変数の違い
- **有効な枝だけを探索する候補提案**と観測データ更新

この例では方式0を選ぶと温度が有効になり、方式1を選ぶと圧力が有効になります。親変数の選択によって探索空間の構造が変わることがポイントです。

## 1. Import と再現性設定

In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.models import (
    HierarchicalConditionalKernelGP,
    HierarchicalConditionalKernelMultiTaskGP,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 通常のカテゴリ変数との違い

Mixed GPでは、カテゴリと連続変数を同時に入力できます。一方、階層型GPでは**親カテゴリの値によって一部の子変数がそもそも意味を持たなくなる**ことをモデル化します。

例えば方式0で圧力を0.2から0.9へ変えても、装置が圧力設定を使わないなら応答は変わらないはずです。このような**条件付き有効性**は、単にカテゴリ列を追加するだけとは異なる構造です。

## 2. 階層構造を定義する

入力を `[x, process_type, temperature, pressure]` とします。

`hierarchical_dependencies` は `{親の列: {親の値: [有効になる子の列]}}` という形式です。

したがって、方式0では温度、方式1では圧力が有効になる構造は次のように表現できます。

In [ ]:
hierarchical_dependencies = {
    1: {
        0: [2],  # process_type == 0 -> temperature が有効
        1: [3],  # process_type == 1 -> pressure が有効
    }
}
hierarchical_dependencies

### 入力列と依存関係の対応

| 列 | 変数 | 方式0 | 方式1 |
|---|---|---|---|
| 0 | 共通変数 `x` | 有効 | 有効 |
| 1 | `process_type`（親） | 0 | 1 |
| 2 | `temperature`（子） | 有効 | 無効 |
| 3 | `pressure`（子） | 無効 | 有効 |

`hierarchical_dependencies={1: {0: [2], 1: [3]}}` は、**列1が0なら列2、1なら列3が有効**という意味です。親変数とtask IDを混同しないでください。

## 3. Synthetic data

共通変数 `x` はどちらの方式でも効きます。方式0では温度、方式1では圧力だけが応答に影響するようにします。

無効な変数にも数値は入っていますが、階層型カーネルではその枝で無効な変数として扱われます。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    x = X[..., 0]
    process_type = X[..., 1]
    temperature = X[..., 2]
    pressure = X[..., 3]

    common = torch.sin(2 * torch.pi * x)
    branch_effect = torch.where(
        process_type == 0,
        0.8 * (temperature - 0.5),
        -0.7 * (pressure - 0.5),
    )
    return (common + branch_effect).unsqueeze(-1)

n_per_branch = 14
x0 = torch.rand(n_per_branch, 1)
x1 = torch.rand(n_per_branch, 1)

X_branch0 = torch.cat([
    x0,
    torch.zeros(n_per_branch, 1),
    torch.rand(n_per_branch, 1),
    torch.rand(n_per_branch, 1),  # この枝では無効
], dim=-1)

X_branch1 = torch.cat([
    x1,
    torch.ones(n_per_branch, 1),
    torch.rand(n_per_branch, 1),  # この枝では無効
    torch.rand(n_per_branch, 1),
], dim=-1)

train_X = torch.cat([X_branch0, X_branch1], dim=0)
train_Y = objective(train_X) + 0.03 * torch.randn(2 * n_per_branch, 1)

print(train_X.shape, train_Y.shape)

### 無効な変数にも値が入る理由

学習データは常に4列のテンソルとして保持するため、無効な子変数にも数値を入れています。ただし、その枝では無効な値として扱うことがモデルの前提です。

候補生成時には無効な列を代表値（ここでは0.5）へ固定すると、同じ物理条件に対して無意味に異なる候補を大量生成することを防げます。

## 4. 単一タスク階層型GP

In [ ]:
model = HierarchicalConditionalKernelGP(
    train_X=train_X,
    train_Y=train_Y,
    hierarchical_dependencies=hierarchical_dependencies,
    use_saas_prior=False,
)

print(type(model).__name__)
print('supports_mll:', model.supports_mll)
print('supports_fit:', model.supports_fit)
print('raw_train_X:', model.raw_train_X.shape)
print('raw_train_Y:', model.raw_train_Y.shape)
print('raw_train_Yvar:', model.raw_train_Yvar)

### 階層型カーネルの役割

通常のカーネルは入力間の距離から類似度を計算します。階層型カーネルは、親条件が異なるときや子変数が無効なときの類似度計算を調整します。

`use_saas_prior=False` は、このNotebookで疎性事前分布を利用しない設定です。まずは階層構造自体の挙動を確認します。

### 学習

階層型GPも robotorchan のexact GP共通インターフェースを使えるため、`fit()` で学習できます。内部では `make_mll()` と BoTorch の `fit_gpytorch_mll()` を再利用しています。

In [ ]:
model.fit()
model.eval()

## 5. 枝ごとのposteriorを比較する

方式0では温度を変化させ、圧力は固定します。方式1では逆に圧力を変化させ、温度を固定します。

### 枝ごとの予測をどう解釈するか

方式0の図では温度だけを、方式1では圧力だけを0から1まで動かしています。横軸は同じ数値範囲でも**物理的には異なる変数**なので、両曲線の差を単純な温度対圧力の効果量として比較しないでください。

以下では平均だけでなく、各枝の事後標準偏差から作る約95%の潜在関数区間も表示します。区間はモデル依存で、観測ノイズ込みの測定値区間ではありません。

In [ ]:
grid = torch.linspace(0.0, 1.0, 101)
x_fixed = torch.full_like(grid, 0.35)

X_test0 = torch.stack([
    x_fixed,
    torch.zeros_like(grid),
    grid,
    torch.full_like(grid, 0.5),
], dim=-1)

X_test1 = torch.stack([
    x_fixed,
    torch.ones_like(grid),
    torch.full_like(grid, 0.5),
    grid,
], dim=-1)

with torch.no_grad():
    post0 = model.posterior(X_test0)
    post1 = model.posterior(X_test1)

mean0 = post0.mean.squeeze(-1)
mean1 = post1.mean.squeeze(-1)

plt.figure(figsize=(8, 4))
plt.plot(grid, mean0, label='Branch 0: temperature active')
plt.plot(grid, mean1, label='Branch 1: pressure active')
plt.xlabel('Active child variable')
plt.ylabel('posterior mean')
plt.legend()
plt.show()

In [ ]:
with torch.no_grad():
    std0 = post0.variance.clamp_min(0).sqrt().squeeze(-1)
    std1 = post1.variance.clamp_min(0).sqrt().squeeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, branch_X, mean, std, title in [
    (axes[0], X_test0, mean0, std0, "Branch 0: temperature active"),
    (axes[1], X_test1, mean1, std1, "Branch 1: pressure active"),
]:
    ax.plot(grid, objective(branch_X).squeeze(-1), "--", label="true objective")
    ax.plot(grid, mean, label="posterior mean")
    ax.fill_between(grid, mean - 1.96 * std, mean + 1.96 * std,
                    alpha=0.2, label="approx. 95% latent interval")
    ax.set_title(title)
    ax.set_xlabel("active child variable")
    ax.legend(fontsize=8)
axes[0].set_ylabel("response")
plt.tight_layout()
plt.show()

### 有効変数と無効変数の違いを可視化する

階層型カーネルの特徴を確認するため、各方式について2種類の操作を比較します。

- **Active sweep**：その方式で有効な子変数だけを0〜1に変える（方式0では温度、方式1では圧力）。
- **Inactive sweep**：有効な子変数を0.5に固定したまま、無効な子変数だけを0〜1に変える。

共通変数 `x=0.35`、親変数の方式は固定します。Active sweep と Inactive sweep は有効な変数の固定値が異なる点にも注意してください。したがって**2本の曲線の高さを直接比較するのではなく、それぞれの曲線がどの程度変化するか**を見ます。

無効な子変数を変えても事後平均がほぼ一定なら、条件付き有効性が予測に反映されていると解釈できます。これは各枝の挙動を可視化する例であり、汎化性能の保証ではありません。

In [ ]:
# Compare sensitivity to active and inactive child variables.
sweep = torch.linspace(0.0, 1.0, 101)
x_common = torch.full_like(sweep, 0.35)
constant = torch.full_like(sweep, 0.5)

# Branch 0: temperature active, pressure inactive.
X_active0 = torch.stack([x_common, torch.zeros_like(sweep), sweep, constant], dim=-1)
X_inactive0 = torch.stack([x_common, torch.zeros_like(sweep), constant, sweep], dim=-1)

# Branch 1: pressure active, temperature inactive.
X_active1 = torch.stack([x_common, torch.ones_like(sweep), constant, sweep], dim=-1)
X_inactive1 = torch.stack([x_common, torch.ones_like(sweep), sweep, constant], dim=-1)

with torch.no_grad():
    active0 = model.posterior(X_active0).mean.squeeze(-1)
    inactive0 = model.posterior(X_inactive0).mean.squeeze(-1)
    active1 = model.posterior(X_active1).mean.squeeze(-1)
    inactive1 = model.posterior(X_inactive1).mean.squeeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, active, inactive, branch in [
    (axes[0], active0, inactive0, "Branch 0 (temperature active)"),
    (axes[1], active1, inactive1, "Branch 1 (pressure active)"),
]:
    ax.plot(sweep, active, label="Active child sweep")
    ax.plot(sweep, inactive, label="Inactive child sweep")
    ax.set_title(branch)
    ax.set_xlabel("Swept child variable")
    ax.legend(fontsize=9)
axes[0].set_ylabel("Posterior mean")
plt.tight_layout()
plt.show()

for branch, active, inactive in [(0, active0, inactive0), (1, active1, inactive1)]:
    print(
        f"Branch {branch} prediction range: "
        f"active={float(active.max() - active.min()):.4g}, "
        f"inactive={float(inactive.max() - inactive.min()):.4g}"
    )

## 6. 階層型Multi-Task GP

次に、同じ階層探索空間を2つの関連タスクで共有します。

ここでは最後の列をtask featureとして追加します。`hierarchical_dependencies` の列番号は **task featureを除いた後の特徴量** に対する番号なので、先ほどと同じ辞書をそのまま使えます。

### Multi-taskとの組合せ

階層構造は**設計空間内の条件付き有効性**を表し、Multi-taskは**関連する複数のタスクから情報を共有すること**を目的とします。これらは独立した概念なので同時に利用できます。

この例ではtask 0とtask 1で同じ階層設計空間を共有し、最後の列にtask IDを追加します。`hierarchical_dependencies` はtask列を除いた4列に対して指定します。

In [ ]:
task0 = torch.zeros(train_X.shape[0], 1)
task1 = torch.ones(train_X.shape[0], 1)

train_X_mt = torch.cat([
    torch.cat([train_X, task0], dim=-1),
    torch.cat([train_X, task1], dim=-1),
], dim=0)

train_Y_mt = torch.cat([
    train_Y,
    0.75 * train_Y + 0.20,
], dim=0)

mt_model = HierarchicalConditionalKernelMultiTaskGP(
    train_X=train_X_mt,
    train_Y=train_Y_mt,
    task_feature=4,
    hierarchical_dependencies=hierarchical_dependencies,
    use_saas_prior=False,
)

print('raw_train_X:', mt_model.raw_train_X.shape)
print('raw_train_Y:', mt_model.raw_train_Y.shape)
print('supports_mll:', mt_model.supports_mll)
print('supports_fit:', mt_model.supports_fit)

In [ ]:
mt_model.fit()
mt_model.eval()

## 7. Multi-Task posterior

方式0・温度0.8という同じ条件について、task 0 / task 1 のposteriorを比較します。

In [ ]:
x_grid = torch.linspace(0.0, 1.0, 101)
base = torch.stack([
    x_grid,
    torch.zeros_like(x_grid),
    torch.full_like(x_grid, 0.8),
    torch.full_like(x_grid, 0.5),
], dim=-1)

X_task0 = torch.cat([base, torch.zeros(len(base), 1)], dim=-1)
X_task1 = torch.cat([base, torch.ones(len(base), 1)], dim=-1)

with torch.no_grad():
    post_task0 = mt_model.posterior(X_task0)
    post_task1 = mt_model.posterior(X_task1)
    mean_task0 = post_task0.mean.squeeze(-1)
    mean_task1 = post_task1.mean.squeeze(-1)
    std_task0 = post_task0.variance.clamp_min(0).sqrt().squeeze(-1)
    std_task1 = post_task1.variance.clamp_min(0).sqrt().squeeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for mean, std, task in [
    (mean_task0, std_task0, "Task 0"),
    (mean_task1, std_task1, "Task 1"),
]:
    axes[0].plot(x_grid, mean, label=task)
    axes[0].fill_between(x_grid, mean - 1.96 * std, mean + 1.96 * std, alpha=0.2)

axes[0].set_title("Task posterior (mean and interval)")
axes[0].set_xlabel("Common variable x")
axes[0].set_ylabel("Response")
axes[0].legend()

axes[1].plot(x_grid, mean_task1 - mean_task0, label="Task 1 - Task 0")
axes[1].axhline(0.0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Difference in posterior means")
axes[1].set_xlabel("Common variable x")
axes[1].set_ylabel("Mean difference")
axes[1].legend()
plt.tight_layout()
plt.show()

### taskごとの予測の違い

左図は、方式0・温度0.8・圧力0.5を固定し、共通変数 `x` を動かしたときの **task 0 / task 1 の予測平均と潜在関数の近似95%事後区間** を示しています。2タスクの曲線の形と水準を比較してください。

右図は同じ入力について **task 1 の予測平均 − task 0 の予測平均** を示しています。正なら task 1 が高く、負なら task 0 が高いことを意味します。右図の差は平均の差であり、**タスク差そのものの不確実性を示す区間ではありません**。差の不確実性を計算する場合は、タスク間の事後共分散が必要です。

なお、task ID は予測対象の違いであり、方式0/1を選ぶ親変数とは異なります。

## 8. 使い分け

### `HierarchicalConditionalKernelGP` が向く場合

- 装置方式によって設定可能なパラメータが変わる
- 材料種によって有効な配合条件が変わる
- アルゴリズム選択によってサブハイパーパラメータが変わる
- 通常の連続・カテゴリ混合だけでは表現しづらい条件付き探索空間

### `HierarchicalConditionalKernelMultiTaskGP` が向く場合

上記に加えて、製品・ライン・装置・測定条件など複数タスク間の相関も同時に利用したい場合に向きます。

### 注意点

`hierarchical_dependencies` は探索空間の構造そのものを表します。単に相関が弱そうな変数を階層化するのではなく、『ある親条件のときだけ子変数に意味がある』という明確な条件付き構造がある場合に使います。

## 9. ベイズ最適化への接続

モデル学習後は通常のBoTorch獲得関数に接続できます。ただし候補生成時にも階層制約を守る必要があります。

たとえば方式0の候補では圧力を無効値に固定し、方式1では温度を固定する、といった候補生成ルールを用意します。探索空間の階層構造と獲得関数最適化の制約は別物なので、両方を整合させることが重要です。

### 階層探索の実務上の注意

学習時に階層型カーネルを使っても、候補生成時に無効な子変数を自由に動かしてよいわけではありません。候補生成側でも枝ごとに有効な変数だけを探索する必要があります。

本Notebookの最後の例は**有限個の候補から事後平均最大の点を選ぶ方法**です。EI/UCBなどの探索を含む獲得関数最適化とは異なります。また、既存例では共通変数 `x` を0.35に固定しているため、4次元全域の最適化ではありません。

## Conditional candidate proposal

各枝で有効な変数だけを変化させ、事後平均が最大の候補を提案します。無効な変数は固定し、枝の制約を保持します。

In [ ]:
with torch.no_grad():
    score0 = model.posterior(X_test0).mean.squeeze(-1)
    score1 = model.posterior(X_test1).mean.squeeze(-1)
    index0 = score0.argmax()
    index1 = score1.argmax()
    candidate0 = X_test0[index0:index0 + 1]
    candidate1 = X_test1[index1:index1 + 1]
    candidate = candidate0 if score0[index0] >= score1[index1] else candidate1
candidate_Y = objective(candidate)
assert candidate.shape == (1, 4)
assert candidate[0, 1].item() in (0.0, 1.0)
print("candidate:", candidate)
print("predicted objective:", model.posterior(candidate).mean.detach())
print("evaluated objective:", candidate_Y)


### 候補提案と実際の評価

`candidate0` と `candidate1` は、それぞれの枝で有効な子変数だけを動かして作った候補です。両枝の予測平均を比較して1点を選び、合成目的関数 `objective` で評価します。

これは**モデル予測による候補選択 → 実際の観測取得**の流れです。現実の実験では `objective` の代わりに装置・シミュレーションなどの評価結果を使います。

## 10. 追加実装例：枝制約を保った観測データ更新

候補を評価したら、新しい観測を学習データへ追加します。方式0では圧力を、方式1では温度を代表値0.5に固定していることを確認します。

更新データを使って次のGPを学習する際には、**同じ `hierarchical_dependencies` を渡す必要があります**。ここでは計算時間を抑えるため再学習は行わず、1回のデータ更新まで示します。

In [ ]:
selected_branch = int(candidate[0, 1].item())
if selected_branch == 0:
    assert torch.isclose(candidate[0, 3], candidate.new_tensor(0.5))
else:
    assert torch.isclose(candidate[0, 2], candidate.new_tensor(0.5))

updated_X = torch.cat([train_X, candidate.detach()], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y.detach()], dim=0)
assert updated_X.shape == (2 * n_per_branch + 1, 4)
assert updated_Y.shape == (2 * n_per_branch + 1, 1)
print("selected process type:", selected_branch)
print("new response:", candidate_Y.item())
print("updated observations:", updated_X.shape, updated_Y.shape)